In [3]:
from pathlib import Path
import pandas as pd
import numpy as np


In [4]:
base = Path(r"C:\Users\LENOVO\Desktop\Kavi docu\jobs\f1-business-analytics")

races = pd.read_csv(base / "data" / "raw" / "races.csv")
drivers = pd.read_csv(base / "data" / "raw" / "drivers.csv")
constructors = pd.read_csv(base / "data" / "raw" / "constructors.csv")
results = pd.read_csv(base / "data" / "raw" / "results.csv")
pit_stops = pd.read_csv(base / "data" / "raw" / "pit_stops.csv")


In [5]:
datasets = [races, drivers, constructors, results, pit_stops]

for df_ in datasets:
    df_.replace(r"\N", np.nan, inplace=True)


In [6]:
for df_ in datasets:
    df_.drop_duplicates(inplace=True)


In [7]:
races["date"] = pd.to_datetime(races["date"], errors="coerce")


In [8]:
drivers["dob"] = pd.to_datetime(drivers["dob"], errors="coerce")
drivers["nationality"] = drivers["nationality"].fillna("Unknown")


In [9]:
constructors["nationality"] = constructors["nationality"].fillna("Unknown")


In [10]:
numeric_cols = ["grid", "position", "points", "laps"]

results[numeric_cols] = (
    results[numeric_cols]
    .apply(pd.to_numeric, errors="coerce")
)

results["points"] = results["points"].fillna(0)
results["position"] = results["position"].fillna(0)
results["grid"] = results["grid"].fillna(0)

# DNF flag
results["dnf"] = (results["position"] == 0).astype(int)


In [11]:
pit_stops["milliseconds"] = pd.to_numeric(
    pit_stops["milliseconds"], errors="coerce"
).fillna(0)


In [12]:
results = results[
    (results["laps"] >= 0) &
    (results["grid"] >= 0)
]


In [13]:
df = (
    results
    .merge(races, on="raceId", how="left", suffixes=("", "_race"))
    .merge(drivers, on="driverId", how="left", suffixes=("", "_driver"))
    .merge(constructors, on="constructorId", how="left", suffixes=("", "_constructor"))
)


In [14]:
drop_cols = [
    "fp1_date","fp1_time","fp2_date","fp2_time",
    "fp3_date","fp3_time","quali_date","quali_time",
    "sprint_date","sprint_time",
    "time","milliseconds","fastestLap","rank",
    "fastestLapTime","fastestLapSpeed",
    "url","url_race","url_driver","url_constructor"
]

df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)


In [15]:
df["driver_name"] = df["forename"] + " " + df["surname"]
df["season"] = df["year"]
df["grid_vs_finish"] = df["grid"] - df["position"]


In [16]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

df[num_cols] = df[num_cols].fillna(0)
df[cat_cols] = df[cat_cols].fillna("Unknown")


In [17]:
print(df.shape)
print(df.isnull().sum().sum())
df.head()


(27238, 32)
0


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,forename,surname,dob,nationality,constructorRef,name_constructor,nationality_constructor,driver_name,season,grid_vs_finish
0,1,18,1,1,22,1.0,1.0,1,1,10.0,...,Lewis,Hamilton,1985-01-07,British,mclaren,McLaren,British,Lewis Hamilton,2008,0.0
1,2,18,2,2,3,5.0,2.0,2,2,8.0,...,Nick,Heidfeld,1977-05-10,German,bmw_sauber,BMW Sauber,German,Nick Heidfeld,2008,3.0
2,3,18,3,3,7,7.0,3.0,3,3,6.0,...,Nico,Rosberg,1985-06-27,German,williams,Williams,British,Nico Rosberg,2008,4.0
3,4,18,4,4,5,11.0,4.0,4,4,5.0,...,Fernando,Alonso,1981-07-29,Spanish,renault,Renault,French,Fernando Alonso,2008,7.0
4,5,18,5,1,23,3.0,5.0,5,5,4.0,...,Heikki,Kovalainen,1981-10-19,Finnish,mclaren,McLaren,British,Heikki Kovalainen,2008,-2.0


In [18]:
df.to_csv(
    base / "data" / "processed" / "f1_clean_analytics_dataset.csv",
    index=False,
    encoding="utf-8"
)
